In [1]:
import os

import pandas as pd
from tqdm import tqdm

In [2]:
PARQUET_DF_PATH = './data/original_data/competition_data_final.parquet'

parquet_parts = list(set(os.listdir(PARQUET_DF_PATH)) - set(['_SUCCESS']))
parquet_parts

['part-00000-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet',
 'part-00008-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet',
 'part-00007-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet',
 'part-00001-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet',
 'part-00005-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet',
 'part-00006-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet',
 'part-00009-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet',
 'part-00004-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet',
 'part-00002-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet',
 'part-00003-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet']

In [ ]:
df = pd.read_parquet(os.path.join(PARQUET_DF_PATH, 'part-00000-aba60f69-2b63-4cc1-95ca-542598094698-c000.snappy.parquet'))

In [17]:
df.head()

,region_name,city_name,cpe_manufacturer_name,cpe_model_name,url_host,cpe_type_cd,cpe_model_os_type,price,date,part_of_day,request_cnt,user_id
0,Краснодарский край,Краснодар,Apple,iPhone 7,ad.adriver.ru,smartphone,iOS,20368.0,2022-06-15,morning,1,45098
1,Краснодарский край,Краснодар,Apple,iPhone 7,apple.com,smartphone,iOS,20368.0,2022-06-19,morning,1,45098
2,Краснодарский край,Краснодар,Apple,iPhone 7,avatars.mds.yandex.net,smartphone,iOS,20368.0,2022-06-12,day,1,45098
3,Краснодарский край,Краснодар,Apple,iPhone 7,googleads.g.doubleclick.net,smartphone,iOS,20368.0,2022-05-16,day,1,45098
4,Краснодарский край,Краснодар,Apple,iPhone 7,googleads.g.doubleclick.net,smartphone,iOS,20368.0,2022-05-30,day,1,45098


In [10]:
number_only_urls = set()

for el in tqdm(df['url_host']):
    if set(el) <= set('0123456789'):
        number_only_urls.add(el)

100%|██████████| 32156858/32156858 [00:56<00:00, 573986.18it/s]


In [17]:
len(number_only_urls)  # note: it's only from one partition

65

In [16]:
list(number_only_urls)[:10]

['40', '34', '16', '43', '33', '19', '12', '59', '15', '127']

In [14]:
df[df['url_host'] == '1'].head()

,region_name,city_name,cpe_manufacturer_name,cpe_model_name,url_host,cpe_type_cd,cpe_model_os_type,price,date,part_of_day,request_cnt,user_id
176834,Ростовская область,Ростов-на-Дону,Samsung,Galaxy A50 Dual,1,smartphone,Android,16408.0,2021-07-21,morning,1,341280
177211,Свердловская область,Екатеринбург,Samsung,Galaxy A50 Dual,1,smartphone,Android,16408.0,2021-06-19,evening,1,341280
359564,Республика Татарстан,Казань,Samsung,Galaxy J6 Plus Dual,1,smartphone,Android,2390.0,2021-07-08,day,1,142005
359763,Республика Татарстан,Казань,Samsung,Galaxy J6 Plus Dual,1,smartphone,Android,2390.0,2021-07-08,morning,1,142005
359808,Республика Татарстан,Казань,Samsung,Galaxy J6 Plus Dual,1,smartphone,Android,2390.0,2021-07-06,evening,1,142005


In [16]:
# public_train = pd.read_parquet('data/original_data/public_train.parquet')

In [6]:
# public_train.head()

,age,is_male,user_id
350459,31.0,1,350459
188276,35.0,1,188276
99002,41.0,0,99002
155506,33.0,0,155506
213873,54.0,0,213873


In [1]:
def is_punycode(s: str) -> bool:
    return s.startswith('xn--')
    
    
def convert_punycode(puny_domain: str) -> str:
    """
    Converts punycode to unicode

    Example:
    >>> convert_punycode('xn--22-glcqfm3bya1b.xn--p1ai')
    <<< 'грузчик22.рф'
    """
    return puny_domain.encode().decode('idna')

In [3]:
s = 'xn--22-glcqfm3bya1b.xn--p1ai'
if is_punycode(s):
    s = convert_punycode(s)
s

'грузчик22.рф'

In [1]:
import pandas as pd

In [13]:
df = pd.read_parquet('data/competition_data_edges.parquet')

In [14]:
df = df.rename(columns = {'url_host_preprocesed': '_orig_url_host_preprocesed'})

In [15]:
df.head()

,user_id,_orig_url_host_preprocesed,request_cnt
0,359133,campaign.aliexpress,4
1,95893,m.facebook,1
2,193478,avr.i-trailer,5
3,348061,e.mail,9
4,316468,sun9-32.userapi,10


In [16]:
df_item_map = pd.read_csv('data/url_host_preprocesed_mapping.csv')

In [17]:
df_item_map.head()

,_orig_url_host_preprocesed,url_host_preprocesed
0,googleads.g.doubleclick,2
1,yandex,3
2,i.ytimg,4
3,vk,5
4,avatars.mds.yandex,6


In [19]:
new_df = df.merge(df_item_map, on=['_orig_url_host_preprocesed'])

In [21]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31740362 entries, 0 to 31740361
Data columns (total 4 columns):
 #   Column                      Dtype 
---  ------                      ----- 
 0   user_id                     int64 
 1   _orig_url_host_preprocesed  object
 2   request_cnt                 int64 
 3   url_host_preprocesed        int64 
dtypes: int64(3), object(1)
memory usage: 968.6+ MB


In [22]:
new_df.isnull().values.any()

False

In [23]:
new_df.head()

,user_id,_orig_url_host_preprocesed,request_cnt,url_host_preprocesed
0,359133,campaign.aliexpress,4,82
1,126106,campaign.aliexpress,1,82
2,88824,campaign.aliexpress,1,82
3,172313,campaign.aliexpress,1,82
4,405304,campaign.aliexpress,1,82


In [24]:
new_df['url_host_preprocesed'].max()

196694

In [2]:
import pandas as pd

In [3]:
age_target_df = pd.read_csv('./data/train_age.csv')

In [6]:
age_target_df['age'].value_counts()

age
2    87270
3    77486
4    42442
1    32641
5    23580
6     5503
Name: count, dtype: int64

In [8]:
age_target_df['age'].value_counts().max() / len(age_target_df)

0.3245178899457835

In [9]:
gender_target_df = pd.read_csv('./data/train_gender.csv')

In [12]:
gender_target_df['is_male'].value_counts()

is_male
1    135332
0    128994
Name: count, dtype: int64

In [5]:
!ls

bin				     lightning_logs
checkpoints			     make_dataset.py
conf				     make_graph.py
create_aggregated_edges.py	     make_url_embedding_outdated.py
create_embedding_tensors.ipynb	     make-url-embeddings.ipynb
create_subdataset_for_debug.ipynb    models
data				     outputs
embeddings_validation_age.work	     playground.ipynb
embeddings_validation_gender.work    preprocess_dataset.py
embeddings_validation_template.yaml  README.md
get_urls.py			     results
hydra_outputs


In [1]:
import os

import torch
from tqdm import tqdm

In [2]:
import pandas as pd

In [47]:
TRAIN_PTLS_PARQUET_PATH = 'data/train_trx_file.parquet'
TEST_PTLS_PARQUET_PATH = 'data/test_trx_file.parquet'

In [48]:
train_partition_paths = [os.path.join(TRAIN_PTLS_PARQUET_PATH, f) for f in os.listdir(TRAIN_PTLS_PARQUET_PATH) if f.endswith('.parquet')]
test_partition_paths = [os.path.join(TEST_PTLS_PARQUET_PATH, f) for f in os.listdir(TEST_PTLS_PARQUET_PATH) if f.endswith('.parquet')]


In [67]:
from typing import List


def check_users_with_repeating_time_exist(partition_paths: List[str]) -> bool:
    for partition_path in tqdm(partition_paths):
        df = pd.read_parquet(partition_path)
        for event_times_of_a_user in df['event_time'].values:
            if len(event_times_of_a_user) != set(event_times_of_a_user):
                return True
    return False


check_users_with_repeating_time_exist(train_partition_paths)

  0%|          | 0/800 [00:00<?, ?it/s]


True

In [49]:
train_all_unique_url_ids = set()
for train_partition_path in tqdm(train_partition_paths):
    df = pd.read_parquet(train_partition_path)
    for urls_of_a_user in df['url_host_preprocesed'].values:
        train_all_unique_url_ids.update(urls_of_a_user)

100%|██████████| 800/800 [00:35<00:00, 22.41it/s]


In [50]:
test_all_unique_url_ids = set()
for test_partition_path in tqdm(test_partition_paths):
    df = pd.read_parquet(test_partition_path)
    for urls_of_a_user in df['url_host_preprocesed'].values:
        test_all_unique_url_ids.update(urls_of_a_user)

100%|██████████| 400/400 [00:03<00:00, 102.99it/s]


In [51]:
max(train_all_unique_url_ids)

196694

In [52]:
max(test_all_unique_url_ids)

196692

In [11]:
set.union(train_all_unique_url_ids, test_all_unique_url_ids) <= set(range(2, 196695))

True

In [53]:
url_encoding_map_df = pd.read_csv('data/url_host_preprocesed_mapping.csv')

In [54]:
all_encodings = url_encoding_map_df['url_host_preprocesed']

In [55]:
set(all_encodings) == set(range(2, 196695))

True

In [56]:
set(all_encodings) == set.union(train_all_unique_url_ids, test_all_unique_url_ids)

False

In [57]:
abscent_urls = set(all_encodings) - set.union(train_all_unique_url_ids, test_all_unique_url_ids)

In [58]:
len(abscent_urls)

1801

In [59]:
abscent_urls

{196609,
 155650,
 163844,
 139279,
 139280,
 163859,
 147476,
 172051,
 180251,
 147484,
 114717,
 155678,
 196639,
 139296,
 196642,
 163878,
 139303,
 155692,
 122951,
 172108,
 147533,
 163934,
 122985,
 163945,
 163950,
 139385,
 139386,
 163965,
 155775,
 155782,
 172167,
 180363,
 147599,
 139419,
 155808,
 131233,
 147620,
 172198,
 131247,
 155835,
 131263,
 139459,
 147653,
 164039,
 172233,
 147661,
 155858,
 106711,
 155868,
 139489,
 180452,
 180460,
 155891,
 155903,
 155905,
 164105,
 147725,
 139537,
 164115,
 139540,
 147745,
 139559,
 180520,
 188711,
 164138,
 155953,
 139575,
 180541,
 180547,
 155974,
 147784,
 147787,
 180560,
 180563,
 115032,
 139612,
 188779,
 147825,
 139644,
 131457,
 115076,
 139658,
 188810,
 188818,
 147859,
 164250,
 172443,
 131485,
 180637,
 164255,
 164256,
 156066,
 131504,
 156084,
 131511,
 188858,
 172498,
 115160,
 115177,
 147951,
 147953,
 156146,
 172531,
 188922,
 147966,
 139784,
 147976,
 131594,
 180745,
 139791,
 188943,
 

In [60]:
url_encoding_map_df[url_encoding_map_df['url_host_preprocesed'].isin(abscent_urls)]

,_orig_url_host_preprocesed,url_host_preprocesed
22704,dot.lipetsk-lmk,22706
36596,kostroma.restate,36598
39320,ulan-ude.restate,39322
48290,irkutsk.vsn,48292
50467,perm.novobyt,50469
...,...,...
196453,polezno4you,196455
196561,vyazma.vsn,196563
196607,morendi-ural,196609
196637,pkuint26,196639


In [19]:
url_host_embeddings_ptls_ids = torch.load('data/url_host_embeddings_ptls_ids.pt')

In [20]:
url_host_embeddings_ptls_ids.shape

torch.Size([196695, 256])

In [21]:
zeros_row = torch.zeros(url_host_embeddings_ptls_ids.shape[1])

In [22]:
# нулевые эмбеддинги только в 0-ом и первом индексе, которые не соответствуют никакому url
(url_host_embeddings_ptls_ids == zeros_row).all(dim=1).nonzero()

tensor([[0],
        [1]])

In [23]:
import sys, os; sys.path.append(os.path.abspath( '..'))

In [24]:

import hydra

In [25]:
hydra.initialize(config_path='conf/trx_custom_embeddings')

/tmp/ipykernel_363467/235621652.py:1: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  hydra.initialize(config_path='conf/trx_custom_embeddings')


hydra.initialize()

In [26]:
cfg = hydra.compose(config_name='ptls_id_to_llm_embedding.yaml', overrides=["+device=cpu"])


In [27]:
d = hydra.utils.instantiate(cfg.url_host_preprocesed)

/path/to/repo/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
d(196694)

tensor([ 1.8617e-02,  1.3645e-01,  5.5538e-02, -2.3758e-02, -3.5959e-02,
        -2.2530e-02, -2.1419e-02,  1.9846e-02, -3.9278e-02,  4.8616e-01,
        -2.9817e-02,  2.4594e-02, -1.9313e-02, -3.7101e-02,  3.4645e-03,
         4.7599e-01,  2.1399e-02, -4.3298e-01, -5.2009e-02, -2.6571e-02,
        -2.3479e-02, -2.5594e-02,  2.6568e-02,  1.8697e-02,  1.4517e-01,
         6.8349e-01, -2.8599e-01,  1.0387e+00,  3.9317e-02,  2.9678e-02,
         2.3859e-02, -2.9296e-02, -1.3630e-01,  7.3004e-03, -2.0036e-02,
        -2.0599e-02,  2.1640e-02, -2.8106e-02, -2.5282e-02, -2.0887e-01,
        -3.6485e-02, -6.1358e-01, -3.1183e-02, -2.1334e-02, -4.9316e-02,
        -9.2740e-02,  2.7009e-01,  4.4680e-02,  2.4289e-02,  2.7969e-02,
        -1.9067e-01,  3.9060e-02, -3.3784e-02, -1.1686e-01,  9.1262e-04,
        -2.5713e-02, -2.0492e-02,  2.8517e-02,  1.8387e+00,  1.5616e-02,
        -1.3533e-03,  2.9584e-02,  2.3886e+00,  1.0335e-02, -4.4284e-02,
        -4.5088e-01, -7.7603e-01,  3.1989e-01,  1.7

In [37]:
url_encoding_map_df[url_encoding_map_df['url_host_preprocesed'] == 196694]

,_orig_url_host_preprocesed,url_host_preprocesed
196692,adminstanovoe,196694


In [44]:
len(d.ptls_item_id_to_pretrained_embedding_id)

199685

In [46]:
d.ptls_item_id_to_pretrained_embedding_id[196694]

tensor(196694)

In [35]:
d

PretrainedEmbeddings(
  (embedding_layer): Embedding(196695, 256)
)

In [43]:
d.ptls_item_id_to_pretrained_embedding_id[123]

tensor(123)